[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-07-interactivity.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Interactivity
**certified-journeys / altair-certified** · Day 7 · Selections & Linked Brushing

> **Goal for today:** Build interactive Altair charts that respond to clicks, brush selections, dropdowns, and legend clicks — and wire two charts together with linked brushing.

In [ ]:
%pip install -q altair vega-datasets

## Step 1 · selection_point — Click-to-Filter on a Scatter Plot

A **`selection_point`** captures single or multi-point clicks. Altair selections are predicates — objects that can be passed directly to `transform_filter()` or used in `condition()` encodings.

| Selection type | Trigger | Common use |
|---|---|---|
| `selection_point` | Click (single/multi) | Highlight one row, legend filter |
| `selection_interval` | Click-drag rectangle | Brush zoom, cross-filter |

Key rule: **define the selection, attach it to a chart with `.add_params()`, then pass it to `transform_filter()` on the same or another chart.**

In [ ]:
import altair as alt
from vega_datasets import data

# Load the cars dataset
cars = data.cars()

# Define a point selection — clicking a point selects it
click_sel = alt.selection_point()

scatter = (
    alt.Chart(cars)
    .mark_point(size=80)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.condition(
            click_sel,          # if selected ...
            alt.Color("Origin:N"),  # ... show full color
            alt.value("lightgray")  # ... else grey
        ),
        tooltip=["Name:N", "Horsepower:Q", "Miles_per_Gallon:Q", "Origin:N"],
    )
    .add_params(click_sel)  # attach selection to the chart
    .properties(title="Click a point to highlight it", width=500, height=300)
)

scatter

**What just happened?**

- `selection_point()` creates an Altair selection object — not a Python variable, but a Vega-Lite predicate.
- `alt.condition(sel, true_encoding, false_encoding)` is the key pattern: selected points get the real color, unselected points go grey.
- **`.add_params(sel)` is required** — the chart won't respond to interactions without it.
- Clicking a point makes it the "active" selection; shift-clicking adds to the selection.

## Step 2 · selection_interval — Brush Rectangle Selection

A **`selection_interval`** creates a draggable rectangle. When used with `transform_filter`, it acts as a live data slice — only points inside the rectangle are passed to downstream charts.

```
brush = alt.selection_interval()
chart.add_params(brush)          # chart captures brush events
other_chart.transform_filter(brush)  # other chart shows only brushed data
```

In [ ]:
# Define an interval (brush) selection
brush = alt.selection_interval()

scatter_brush = (
    alt.Chart(cars)
    .mark_point(size=60)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.condition(brush, alt.Color("Origin:N"), alt.value("lightgray")),
        opacity=alt.condition(brush, alt.value(1.0), alt.value(0.2)),
    )
    .add_params(brush)
    .properties(title="Drag to brush — selected points stay vivid", width=500, height=300)
)

scatter_brush

**What just happened?**

- **Drag** on the chart to draw a rectangle; points inside are "selected".
- We use `opacity` as a second conditional encoding — unselected points fade to 20% opacity.
- **The brush itself is an empty selection** until you drag; all points start highlighted.
- Clear the selection by clicking outside the brush rectangle.

## Step 3 · transform_filter — Filter a Second Chart by the Brush

The power of Altair selections is **cross-chart filtering**: attach the selection to Chart A, then call `transform_filter(selection)` on Chart B. Chart B will only render the rows where the selection predicate is true.

> **Always chain `transform_filter(selection)` after the selection definition** — the selection is passed as a predicate, not a DataFrame slice.

In [ ]:
brush2 = alt.selection_interval()

# Top chart: scatter with brush — captures user interaction
top = (
    alt.Chart(cars)
    .mark_point(size=60)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.condition(brush2, alt.Color("Origin:N"), alt.value("lightgray")),
    )
    .add_params(brush2)  # brush lives HERE
    .properties(title="Brush on the scatter to filter the bar chart below", width=500, height=250)
)

# Bottom chart: bar chart filtered by brush — transform_filter receives the selection
bottom = (
    alt.Chart(cars)
    .mark_bar()
    .encode(
        x=alt.X("Origin:N"),
        y=alt.Y("count():Q"),
        color=alt.Color("Origin:N"),
    )
    .transform_filter(brush2)  # only rows inside the brush are counted
    .properties(title="Count of brushed cars by origin", width=500, height=150)
)

# Stack vertically with &
top & bottom

**What just happened?**

- `brush2` is attached to `top` — that chart generates the selection events.
- `transform_filter(brush2)` on `bottom` tells Vega-Lite: "only render rows that pass the brush predicate".
- **The bar chart updates in real time** as you drag the brush.
- `top & bottom` is shorthand for `alt.vconcat(top, bottom)` — stacks charts vertically.

## Step 4 · Legend-Click Filtering with bind='legend'

You can bind a `selection_point` directly to the legend so clicking a legend item filters the chart. Pass `bind='legend'` and set `fields` to the column the legend encodes.

This is a common UX pattern for categorical filters without any extra UI controls.

In [ ]:
# Legend-click selection — binds to the 'Origin' color legend
legend_sel = alt.selection_point(fields=["Origin"], bind="legend")

legend_chart = (
    alt.Chart(cars)
    .mark_point(size=80, filled=True)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.Color("Origin:N"),
        # opacity reacts to the legend selection
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.1)),
        size=alt.condition(legend_sel, alt.value(80), alt.value(30)),
    )
    .add_params(legend_sel)
    .properties(
        title="Click a legend item to filter by Origin",
        width=500, height=300
    )
)

legend_chart

**What just happened?**

- `bind='legend'` wires the selection directly to the chart's color legend — **no extra widget needed**.
- `fields=['Origin']` tells Altair which column the legend click filters on.
- We apply two conditional encodings (opacity + size) for a clear visual effect.
- **Shift-click** multiple legend items to keep several origins visible simultaneously.

## Step 5 · Dropdown Filter with bind_select

`bind_select` connects an HTML `<select>` dropdown to a selection parameter. You define the selection with `alt.selection_point(bind=alt.binding_select(options=[...]))`, then use `transform_filter` to slice the data.

This renders as a native HTML dropdown rendered above (or next to) the chart — no external widget library needed.

In [ ]:
# Get unique cylinder values for the dropdown
cylinders = sorted(cars["Cylinders"].dropna().unique().tolist())

# Build a binding: HTML select widget with a label
cyl_binding = alt.binding_select(
    options=[None] + cylinders,  # None = "show all"
    labels=["All"] + [str(c) for c in cylinders],
    name="Cylinders: "
)

# selection_point bound to the widget; filter on 'Cylinders' field
cyl_sel = alt.selection_point(
    fields=["Cylinders"],
    bind=cyl_binding
)

dropdown_chart = (
    alt.Chart(cars)
    .mark_point(size=80, filled=True)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.Color("Origin:N"),
        tooltip=["Name:N", "Cylinders:Q", "Horsepower:Q"],
    )
    .transform_filter(cyl_sel)  # rows not matching the selected cylinder are hidden
    .add_params(cyl_sel)
    .properties(title="Filter by cylinder count via dropdown", width=500, height=300)
)

dropdown_chart

**What just happened?**

- `alt.binding_select(options=...)` creates the HTML dropdown — Altair renders it automatically above the chart.
- `options=[None] + cylinders` with matching labels lets the user pick "All" to remove the filter.
- **`transform_filter(cyl_sel)` removes non-matching rows entirely** — unlike `condition()`, the filtered-out points disappear rather than greying out.
- Other binding types: `alt.binding_range` (slider), `alt.binding_checkbox`, `alt.binding_radio`.

## Step 6 · Conditional Encoding — Highlight vs Grey-Out

`alt.condition(selection, true_value, false_value)` works inside any encoding channel. You can combine multiple conditions and encodings for rich visual feedback: highlighted points can be colored, sized, and made opaque while background points are small, grey, and translucent.

In [ ]:
multi_sel = alt.selection_point()

highlight_chart = (
    alt.Chart(cars)
    .mark_point(stroke="black")
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        # Color: full color when selected, grey when not
        color=alt.condition(
            multi_sel,
            alt.Color("Cylinders:O", scale=alt.Scale(scheme="viridis")),
            alt.value("#d3d3d3")
        ),
        # Size: bigger when selected
        size=alt.condition(multi_sel, alt.value(150), alt.value(30)),
        # Opacity: fully visible when selected, faint when not
        opacity=alt.condition(multi_sel, alt.value(1.0), alt.value(0.25)),
        # Stroke width: bold outline when selected
        strokeWidth=alt.condition(multi_sel, alt.value(1.5), alt.value(0)),
        tooltip=["Name:N", "Cylinders:Q", "Horsepower:Q", "Miles_per_Gallon:Q"],
    )
    .add_params(multi_sel)
    .properties(
        title="Click to highlight — shift-click for multi-select",
        width=550, height=320
    )
)

highlight_chart

**What just happened?**

- Four encoding channels (`color`, `size`, `opacity`, `strokeWidth`) all use `alt.condition` simultaneously.
- The `true_value` in each condition can be a full encoding spec (like `alt.Color(...)`) or a literal `alt.value(...)`.
- **Shift-click** to add points to the multi-selection; plain click resets to one point.
- This pattern is more visually informative than simply hiding points — context is preserved.

## Step 7 · Two-Panel Linked Brushing Dashboard

A classic interactive analytics pattern: **brush the scatter to update the bar chart**. One selection, two charts, one `|` combinator.

The brush is defined once, attached to the scatter, and passed to `transform_filter` on the bar chart. Both charts share the same underlying dataset.

In [ ]:
brush_dash = alt.selection_interval(encodings=["x"])  # brush only along x-axis

# Left panel: scatter — brush source
scatter_panel = (
    alt.Chart(cars)
    .mark_point(size=60, filled=True)
    .encode(
        x=alt.X("Horsepower:Q", scale=alt.Scale(zero=False)),
        y=alt.Y("Miles_per_Gallon:Q", scale=alt.Scale(zero=False)),
        color=alt.condition(brush_dash, alt.Color("Origin:N"), alt.value("lightgray")),
        opacity=alt.condition(brush_dash, alt.value(0.9), alt.value(0.15)),
    )
    .add_params(brush_dash)  # scatter owns the brush
    .properties(title="Brush a horsepower range", width=380, height=280)
)

# Right panel: bar chart — filtered by brush
bar_panel = (
    alt.Chart(cars)
    .mark_bar()
    .encode(
        x=alt.X(
            "mean(Miles_per_Gallon):Q",
            title="Avg MPG",
            scale=alt.Scale(domain=[0, 50])
        ),
        y=alt.Y("Origin:N", sort="-x"),
        color=alt.Color("Origin:N"),
        tooltip=["Origin:N", "mean(Miles_per_Gallon):Q"],
    )
    .transform_filter(brush_dash)  # average MPG only for brushed cars
    .properties(title="Avg MPG for brushed range", width=280, height=280)
)

# Combine side-by-side using |
dashboard = scatter_panel | bar_panel
dashboard.properties(title="Linked Brushing: Horsepower vs MPG")

**What just happened?**

- `encodings=['x']` restricts the brush to only the x-axis — easier to select a horsepower range.
- The bar chart shows **average MPG per origin** for only the brushed cars.
- `scatter_panel | bar_panel` is shorthand for `alt.hconcat(scatter_panel, bar_panel)`.
- **Debug tip:** build and test each panel in isolation before composing — errors in one panel can obscure the other.

In [ ]:
# Challenge: Build a two-panel linked dashboard using the 'gapminder' dataset
# Panel 1: scatter plot of fertility vs life_expect, colored by cluster
#           with a selection_interval brush
# Panel 2: bar chart showing mean(life_expect) by region,
#           filtered to only the brushed countries
# Requirements:
#   - Use alt.selection_interval() attached to Panel 1
#   - Panel 2 uses transform_filter(brush)
#   - Combine with | operator
#   - Add a title to the whole composed chart

from vega_datasets import data as vd

gap = vd.gapminder()
# Inspect available columns:
# print(gap.columns.tolist())
# print(gap.head(2))

# TODO: define your brush selection
# gap_brush = alt.selection_interval(...)

# TODO: build panel 1 (scatter: fertility vs life_expect)
# panel1 = (
#     alt.Chart(gap)
#     ...
#     .add_params(gap_brush)
#     .properties(width=380, height=280)
# )

# TODO: build panel 2 (bar: mean life_expect by cluster, filtered)
# panel2 = (
#     alt.Chart(gap)
#     ...
#     .transform_filter(gap_brush)
#     .properties(width=280, height=280)
# )

# TODO: compose and display
# (panel1 | panel2).properties(title="Gapminder Linked Brush")

print("Gap columns:", gap.columns.tolist())

---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `selection_point()` | Click-based selection; use with `alt.condition()` for highlight/grey |
| `selection_interval()` | Drag-rectangle brush; restrict with `encodings=['x']` for 1-D brush |
| `transform_filter(sel)` | Passes selection as a predicate — filtered rows disappear entirely |
| `bind='legend'` | Wires selection to chart legend — no extra widget needed |
| `binding_select(...)` | Dropdown widget; `options=[None]` + label `"All"` for "show all" |
| `alt.condition(sel, true, false)` | Works in any encoding channel — can combine multiple channels |
| `panel1 \| panel2` | `alt.hconcat()` shorthand for side-by-side linked charts |

> **Tip:** Selections become filters via `transform_filter(selection)` — always chain it after the selection definition. The selection object is a predicate, not a dataframe slice.

---
## What's next
**Day 8** → Customization & Themes — apply built-in themes, configure axes, legends, responsive widths, and build a reusable custom theme function.

Mark Day 7 complete in your [tracker](../index.html).